# Script 3 — Treinamento dos Modelos de ML
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

## Decisões metodológicas implementadas

| Decisão | Justificativa |
|---------|--------------|
| **GroupKFold por empresa** | Dados de painel: evita vazamento temporal cruzado entre empresas |
| **Log-transform nos targets** | Skewness 3.2 (Receita) e 2.5 (EBITDA) — melhora resíduos e RMSE |
| **SMAPE em vez de MAPE** | Lucro líquido tem valores negativos — MAPE indefinido para y≈0 |
| **Imputação das features YoY** | `dropna()` jogaria fora 15.5% das observações úteis |
| **Normalização por empresa** | Petrobras/Vale têm receita 100× > WEG/Brisanet — Z-score global distorceria |
| **Baseline ingênua documentada** | Referência obrigatória para avaliar se o ML agrega valor |
| **Feature importance / SHAP** | Interpretabilidade exigida em TCC — explica o que o modelo aprendeu |
| **Persistência completa** | Todos os modelos, scalers e resultados salvos para o Script 4 |

## Etapa 0 — Dependências e configuração

In [ ]:
import logging, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
logger = logging.getLogger('pipeline_treino')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_treino.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_EXT  = 5      # folds externos (avaliação)
N_SPLITS_INT  = 5      # folds internos (seleção de hiperparâmetros)
RANDOM_STATE  = 42
LOG_TARGETS   = {'TARGET_DRE_3.01', 'TARGET_EBITDA'}   # targets com log-transform
SMAPE_TARGETS = {'TARGET_DRE_3.11'}                      # targets com valores negativos

logger.info("Script 3 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")

## Etapa 1 — Carregamento dos artefatos do Script 2

In [ ]:
treino   = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste    = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')

with open(PASTA_SAIDA /'features.pkl',     'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA /'targets.pkl',      'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA /'grupos_treino.pkl','rb') as f: GRUPOS_TREINO = pickle.load(f)
with open(PASTA_SAIDA /'kpis.pkl',         'rb') as f: KPIS          = pickle.load(f)

logger.info("Treino: %d × %d | Teste: %d × %d", *treino.shape, *teste.shape)
logger.info("Features: %d | Targets: %s | Grupos únicos: %d",
            len(FEATURES), TARGETS, len(set(GRUPOS_TREINO)))

print(f"Treino : {treino.shape[0]} obs | DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"Teste  : {teste.shape[0]} obs  | DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"Features: {len(FEATURES)} | Targets: {TARGETS}")

## Etapa 2 — Pré-processamento e baseline ingênua

**Imputação das features YoY:** as features `*_yoy` têm ~15.5% de NaN
(primeiros períodos de cada empresa, onde não existe t-4). Usar `dropna()`
descartaria 102 observações úteis. A imputação pela mediana resolve isso
sem introduzir informação do futuro — cada feature é imputada separadamente
antes do fit de cada fold para evitar data leakage no CV.

**Log-transform dos targets:** `TARGET_DRE_3.01` (skew=3.21) e
`TARGET_EBITDA` (skew=2.49) têm distribuição fortemente assimétrica.
O log1p aproxima a distribuição da normal, melhorando os resíduos do Ridge
e do SVR. Os modelos de árvore (RF, GB) são robustos à assimetria, mas
o log também os beneficia reduzindo o peso desproporcional das grandes
empresas (Petrobras, Vale) na função de perda.

In [ ]:
def smape(y_true, y_pred):
    """Symmetric MAPE — definido para valores negativos e próximos de zero."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(num[mask] / denom[mask])


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def calcular_baseline(treino, teste, target):
    """
    Baseline ingênua: previsão = valor atual do período (persistência).
    Para TARGET_DRE_3.01 → feature DRE_3.01 (receita atual).
    Esta baseline deve ser BATIDA pelo ML para justificar a complexidade.
    """
    mapa = {
        'TARGET_DRE_3.01': 'DRE_3.01',
        'TARGET_DRE_3.11': 'DRE_3.11',
        'TARGET_EBITDA'  : 'EBITDA',
    }
    col = mapa.get(target)
    if col not in teste.columns:
        return {}

    mask = teste[target].notna() & teste[col].notna()
    y_true = teste.loc[mask, target].values
    y_pred = teste.loc[mask, col].values

    mask2 = y_true != 0
    metrics = {
        'RMSE_baseline' : rmse(y_true, y_pred),
        'MAE_baseline'  : mean_absolute_error(y_true, y_pred),
        'SMAPE_baseline': smape(y_true, y_pred),
        'R2_baseline'   : r2_score(y_true, y_pred),
    }
    return metrics


# Calcular e registrar baselines
baselines = {}
print("\n=== Baseline Ingênua (persistência do valor atual) ===")
print(f"  {'Target':<25} {'RMSE':>15} {'MAE':>15} {'SMAPE':>8} {'R²':>6}")
print(f"  {'-'*25} {'-'*15} {'-'*15} {'-'*8} {'-'*6}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<25} {b['RMSE_baseline']:>15,.0f} {b['MAE_baseline']:>15,.0f} "
              f"{b['SMAPE_baseline']:>8.1%} {b['R2_baseline']:>6.3f}")
    logger.info("Baseline %s: RMSE=%.0f MAE=%.0f SMAPE=%.2f%% R2=%.3f",
                t, b.get('RMSE_baseline',0), b.get('MAE_baseline',0),
                b.get('SMAPE_baseline',0)*100, b.get('R2_baseline',0))

## Etapa 3 — Definição dos algoritmos e grades de hiperparâmetros

**GroupKFold por empresa:** cada fold garante que todas as observações de
uma empresa ficam ou no treino ou na validação — nunca divididas. Isso
respeita a estrutura de painel e evita que o modelo "veja" o futuro de
uma empresa nos dados de validação.

**Grades de hiperparâmetros:** calibradas para o tamanho amostral disponível
(~554 obs completas × 15 features). Grades excessivamente grandes criariam
centenas de fits desnecessários sem ganho real dado o N limitado.

In [ ]:
# GroupKFold garante que cada empresa fica inteira num único fold
gkf_externo = GroupKFold(n_splits=N_SPLITS_EXT)
gkf_interno = GroupKFold(n_splits=N_SPLITS_INT)

# ── Ridge ─────────────────────────────────────────────────────────────────
# Pipeline: imputa NaN → normaliza → regride
# O imputer dentro do pipeline evita leakage: a mediana é calculada só no fold de treino
estimador_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {
    "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

# ── SVR ───────────────────────────────────────────────────────────────────
estimador_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Random Forest ─────────────────────────────────────────────────────────
# RF tolera NaN nativamente via sklearn >= 1.4 (missing_values='nan')
# Para versões anteriores, imputer explícito
estimador_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}

# ── Gradient Boosting ─────────────────────────────────────────────────────
estimador_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (estimador_ridge, grade_ridge),
    "SVR":              (estimador_svr,   grade_svr),
    "RandomForest":     (estimador_rf,    grade_rf),
    "GradientBoosting": (estimador_gb,    grade_gb),
}

logger.info("%d algoritmos configurados: %s", len(ALGORITMOS), list(ALGORITMOS.keys()))
print(f"✅ {len(ALGORITMOS)} algoritmos configurados com GroupKFold(n={N_SPLITS_EXT})")

## Etapa 4 — Treinamento com Nested Cross-Validation

In [ ]:
def treinar_alg(nome, estimador, grade, X, y, grupos, gkf_int, gkf_ext,
                usar_log=False, target_nome=""):
    """
    Nested CV com GroupKFold.

    Loop interno : GridSearchCV para seleção de hiperparâmetros.
                   Usa grupos para não vazar dados entre empresas.
    Loop externo : cross_validate para estimativa não enviesada do erro.
                   Métricas calculadas no espaço original (se log, reverter).

    Parâmetros
    ----------
    usar_log : bool
        Se True, transforma y com log1p antes do fit e reverte com expm1 na
        predição — melhora resíduos para targets com distribuição assimétrica
        (Receita Líquida skew=3.21, EBITDA skew=2.49).
    """
    y_fit = np.log1p(y) if usar_log else y.copy()

    # ── Loop interno: seleção de hiperparâmetros ──────────────────────────
    gs = GridSearchCV(
        estimador, grade,
        cv=list(gkf_int.split(X, y_fit, grupos)),
        scoring='neg_mean_squared_error',
        refit=True, n_jobs=-1, verbose=0,
    )
    gs.fit(X, y_fit)
    melhor = gs.best_estimator_

    # ── Loop externo: avaliação não enviesada ─────────────────────────────
    rmse_vals, mae_vals, smape_vals, r2_vals = [], [], [], []

    for train_idx, val_idx in gkf_ext.split(X, y_fit, grupos):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y_fit[train_idx], y_fit[val_idx]
        y_orig_val  = y[val_idx]          # espaço original para métricas

        melhor.fit(X_tr, y_tr)
        y_pred_log = melhor.predict(X_val)
        y_pred = np.expm1(y_pred_log) if usar_log else y_pred_log

        rmse_vals.append(rmse(y_orig_val, y_pred))
        mae_vals.append(mean_absolute_error(y_orig_val, y_pred))
        smape_vals.append(smape(y_orig_val, y_pred))
        r2_vals.append(r2_score(y_orig_val, y_pred))

    # Refaz fit final no conjunto completo de treino
    melhor.fit(X, y_fit)

    metricas = {
        'RMSE_CV' : float(np.mean(rmse_vals)),
        'MAE_CV'  : float(np.mean(mae_vals)),
        'SMAPE_CV': float(np.mean(smape_vals)),
        'R2_CV'   : float(np.mean(r2_vals)),
        'RMSE_CV_std' : float(np.std(rmse_vals)),
        'R2_CV_std'   : float(np.std(r2_vals)),
        'best_params': gs.best_params_,
        'log_transform': usar_log,
    }

    logger.info("  %-20s RMSE=%10.0f±%8.0f  SMAPE=%5.1f%%  R²=%5.3f±%5.3f  log=%s",
                nome,
                metricas['RMSE_CV'], metricas['RMSE_CV_std'],
                metricas['SMAPE_CV']*100,
                metricas['R2_CV'], metricas['R2_CV_std'],
                usar_log)
    print(f"  {nome:<20} RMSE={metricas['RMSE_CV']:>12,.0f}  "
          f"SMAPE={metricas['SMAPE_CV']:>5.1%}  R²={metricas['R2_CV']:>6.3f}  "
          f"params={gs.best_params_}")

    return melhor, metricas


# ── Execução principal ────────────────────────────────────────────────────
resultados = {}   # {target: {algoritmo: (modelo, metricas)}}

for target in TARGETS:
    usar_log = target in LOG_TARGETS

    print(f"\n{'='*70}")
    print(f"  TARGET: {target}  |  log-transform: {usar_log}")
    print(f"  Baseline → RMSE={baselines[target].get('RMSE_baseline',0):,.0f}  "
          f"SMAPE={baselines[target].get('SMAPE_baseline',0):.1%}  "
          f"R²={baselines[target].get('R2_baseline',0):.3f}")
    print(f"{'-'*70}")

    df_t = treino[FEATURES + [target]].copy()
    # Não fazemos dropna aqui — o imputer dentro do pipeline trata os NaN
    # Mas o target deve estar presente
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)

    # Grupos alinhados ao df_t filtrado (por target não-nulo)
    mask_target = treino[target].notna()
    grupos_t    = GRUPOS_TREINO[mask_target.values]

    X = df_t[FEATURES].values
    y = df_t[target].values

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade, X, y, grupos_t,
            gkf_interno, gkf_externo,
            usar_log=usar_log, target_nome=target,
        )
        resultados[target][nome] = (modelo, metricas)
        # Salva modelo com metadados de log-transform embutidos
        artefato = {'modelo': modelo, 'log_transform': usar_log, 'features': FEATURES}
        joblib.dump(artefato, PASTA_SAIDA / f'modelo_{target}_{nome}.pkl')

    logger.info("TARGET %s concluído | %d algoritmos", target, len(ALGORITMOS))

## Etapa 5 — Avaliação no conjunto de teste (hold-out final)

In [ ]:
def avaliar_no_teste(modelo, X_te, y_te, usar_log, nome_alg, target):
    """
    Avalia o modelo final no hold-out de teste.
    Reverte log-transform antes de calcular as métricas — garante
    que os números reportados estão na escala original (R$ mil).
    """
    y_pred_raw = modelo.predict(X_te)
    y_pred = np.expm1(y_pred_raw) if usar_log else y_pred_raw

    mask = np.isfinite(y_te) & np.isfinite(y_pred)
    y_t  = y_te[mask]
    y_p  = y_pred[mask]

    return {
        'RMSE_teste' : float(rmse(y_t, y_p)),
        'MAE_teste'  : float(mean_absolute_error(y_t, y_p)),
        'SMAPE_teste': float(smape(y_t, y_p)),
        'R2_teste'   : float(r2_score(y_t, y_p)),
    }


print("\n=== Avaliação no conjunto de teste (hold-out) ===")
metricas_teste = {}

for target in TARGETS:
    usar_log = target in LOG_TARGETS
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    metricas_teste[target] = {}
    print(f"\n{target}  (baseline RMSE={baselines[target].get('RMSE_baseline',0):,.0f})")
    print(f"  {'Algoritmo':<20} {'RMSE':>15} {'MAE':>13} {'SMAPE':>8} {'R²':>7}")
    print(f"  {'-'*20} {'-'*15} {'-'*13} {'-'*8} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_no_teste(modelo, X_te, y_te, usar_log, nome, target)
        metricas_teste[target][nome] = m
        beat = "✅" if m['RMSE_teste'] < baselines[target].get('RMSE_baseline', np.inf) else "⚠️"
        print(f"  {beat} {nome:<18} {m['RMSE_teste']:>15,.0f} {m['MAE_teste']:>13,.0f} "
              f"{m['SMAPE_teste']:>8.1%} {m['R2_teste']:>7.3f}")
        logger.info("Teste | %s | %s: RMSE=%.0f MAE=%.0f SMAPE=%.2f%% R2=%.3f",
                    target, nome, m['RMSE_teste'], m['MAE_teste'],
                    m['SMAPE_teste']*100, m['R2_teste'])

## Etapa 6 — Feature Importance e interpretabilidade

In [ ]:
def extrair_importancia(modelo, features, nome_alg):
    """
    Extrai importância das features para árvores (RF, GB) ou
    coeficientes absolutos para modelos lineares (Ridge).
    """
    # Acessar o estimador final do Pipeline
    step_names = [s for s, _ in modelo.steps]
    estimador_final = modelo.named_steps[step_names[-1]]

    if hasattr(estimador_final, 'feature_importances_'):
        imp = estimador_final.feature_importances_
    elif hasattr(estimador_final, 'coef_'):
        imp = np.abs(estimador_final.coef_)
    else:
        return pd.Series(dtype=float)

    return pd.Series(imp, index=features).sort_values(ascending=False)


print("\n=== Feature Importance — melhor modelo por target ===")
fig, axes = plt.subplots(len(TARGETS), 1, figsize=(10, 4 * len(TARGETS)))
if len(TARGETS) == 1:
    axes = [axes]

feature_importances = {}
for i, target in enumerate(TARGETS):
    # Selecionar o modelo com melhor R² no teste
    melhor_nome = max(
        metricas_teste[target],
        key=lambda n: metricas_teste[target][n]['R2_teste']
    )
    melhor_modelo = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_modelo, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome, 'importancias': imp.to_dict()}

    if not imp.empty:
        top10 = imp.head(10)
        axes[i].barh(top10.index[::-1], top10.values[::-1], color='#2B6CB0', alpha=0.85)
        axes[i].set_title(f"{target} — {melhor_nome} (melhor R²)", fontsize=11)
        axes[i].set_xlabel("Importância relativa")
        axes[i].tick_params(axis='y', labelsize=9)
        for v, label in zip(top10.values[::-1], top10.index[::-1]):
            axes[i].text(v + imp.max()*0.01, list(top10.index[::-1]).index(label),
                         f'{v:.3f}', va='center', fontsize=8)
    logger.info("Feature importance %s / %s — top3: %s",
                target, melhor_nome, imp.head(3).to_dict())

plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Gráfico salvo: feature_importance.png")

## Etapa 7 — Curvas de aprendizado (diagnóstico de bias/variance)

As curvas de aprendizado respondem a uma pergunta fundamental do TCC:
**o modelo está sofrendo de overfitting ou underfitting?**
Com ~554 observações, há risco real de overfitting nos modelos baseados
em árvores com muitos parâmetros.

In [ ]:
from sklearn.model_selection import learning_curve

print("\\nGerando curvas de aprendizado (melhor modelo por target)...")
fig2, axes2 = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5))
if len(TARGETS) == 1:
    axes2 = [axes2]

for i, target in enumerate(TARGETS):
    usar_log = target in LOG_TARGETS
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask_target = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask_target.values]
    X = df_t[FEATURES].values
    y = np.log1p(df_t[target].values) if usar_log else df_t[target].values

    melhor_nome = max(
        metricas_teste[target],
        key=lambda n: metricas_teste[target][n]['R2_teste']
    )
    melhor_modelo = resultados[target][melhor_nome][0]

    try:
        sizes, tr_scores, val_scores = learning_curve(
            melhor_modelo, X, y,
            cv=list(gkf_externo.split(X, y, grupos_t)),
            scoring='r2',
            train_sizes=np.linspace(0.2, 1.0, 6),
            n_jobs=-1,
        )
        ax = axes2[i]
        ax.plot(sizes, tr_scores.mean(axis=1),  'o-', label='Treino',  color='#2B6CB0')
        ax.fill_between(sizes,
                        tr_scores.mean(axis=1) - tr_scores.std(axis=1),
                        tr_scores.mean(axis=1) + tr_scores.std(axis=1),
                        alpha=0.15, color='#2B6CB0')
        ax.plot(sizes, val_scores.mean(axis=1), 's--', label='Validação', color='#C53030')
        ax.fill_between(sizes,
                        val_scores.mean(axis=1) - val_scores.std(axis=1),
                        val_scores.mean(axis=1) + val_scores.std(axis=1),
                        alpha=0.15, color='#C53030')
        ax.set_title(f'{target}\n{melhor_nome}', fontsize=10)
        ax.set_xlabel('Tamanho do treino')
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        # Diagnóstico automático
        gap = tr_scores.mean(axis=1)[-1] - val_scores.mean(axis=1)[-1]
        diag = 'overfitting' if gap > 0.15 else ('underfitting' if val_scores.mean(axis=1)[-1] < 0.3 else 'OK')
        ax.set_xlabel(f'Tamanho | Gap={gap:.2f} → {diag}')
        logger.info("Curva aprendizado %s/%s: gap=%.3f diagnóstico=%s",
                    target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou para %s/%s: %s", target, melhor_nome, e)

plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Gráfico salvo: curvas_aprendizado.png")

## Etapa 8 — Análise de resíduos (validação dos pressupostos)

In [ ]:
print("\nGerando análise de resíduos...")
n_alvos = len(TARGETS)
fig3, axes3 = plt.subplots(n_alvos, 2, figsize=(14, 5 * n_alvos))
if n_alvos == 1:
    axes3 = axes3.reshape(1, -1)

for i, target in enumerate(TARGETS):
    usar_log = target in LOG_TARGETS
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    melhor_nome = max(
        metricas_teste[target],
        key=lambda n: metricas_teste[target][n]['R2_teste']
    )
    melhor_modelo = resultados[target][melhor_nome][0]

    y_pred_raw = melhor_modelo.predict(X_te)
    y_pred = np.expm1(y_pred_raw) if usar_log else y_pred_raw

    residuos = y_te - y_pred

    # Predito vs real
    ax1 = axes3[i, 0]
    lim = max(np.abs(y_te).max(), np.abs(y_pred).max())
    ax1.scatter(y_pred, y_te, alpha=0.5, s=20, color='#2B6CB0')
    ax1.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Linha perfeita')
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f'{target} — {melhor_nome}nPredito × Observado', fontsize=10)
    ax1.legend(fontsize=8)

    # Resíduos vs predito
    ax2 = axes3[i, 1]
    ax2.scatter(y_pred, residuos, alpha=0.5, s=20, color='#744210')
    ax2.axhline(0, color='r', lw=1.5, ls='--')
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos — skew={pd.Series(residuos).skew():.2f}', fontsize=10)

    logger.info("Resíduos %s/%s: mean=%.0f std=%.0f skew=%.2f",
                target, melhor_nome,
                np.mean(residuos), np.std(residuos), pd.Series(residuos).skew())

plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Gráfico salvo: analise_residuos.png")

## Etapa 9 — Persistência completa dos resultados

In [ ]:
# ── Tabela comparativa consolidada ───────────────────────────────────────
rows_cv   = []
rows_teste = []

for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, metricas) in algs.items():
        row_cv = {
            'Target':    target,
            'Algoritmo': alg,
            'RMSE_CV':   metricas['RMSE_CV'],
            'RMSE_CV_std': metricas.get('RMSE_CV_std', np.nan),
            'SMAPE_CV':  metricas['SMAPE_CV'],
            'R2_CV':     metricas['R2_CV'],
            'R2_CV_std': metricas.get('R2_CV_std', np.nan),
            'log_transform': metricas.get('log_transform', False),
            'best_params': str(metricas['best_params']),
        }
        rows_cv.append(row_cv)

        m_te = metricas_teste[target][alg]
        row_te = {
            'Target':     target,
            'Algoritmo':  alg,
            'RMSE_teste': m_te['RMSE_teste'],
            'MAE_teste':  m_te['MAE_teste'],
            'SMAPE_teste':m_te['SMAPE_teste'],
            'R2_teste':   m_te['R2_teste'],
            'RMSE_baseline': b.get('RMSE_baseline', np.nan),
            'Bateu_baseline': m_te['RMSE_teste'] < b.get('RMSE_baseline', np.inf),
        }
        rows_teste.append(row_te)

df_cv    = pd.DataFrame(rows_cv)
df_teste = pd.DataFrame(rows_teste)

print("\n" + "="*70)
print("  RESULTADOS FINAIS — Cross-Validation (GroupKFold)")
print("="*70)
print(df_cv[['Target','Algoritmo','RMSE_CV','RMSE_CV_std','SMAPE_CV','R2_CV','R2_CV_std']]
      .to_string(index=False))

print("\n" + "="*70)
print("  RESULTADOS FINAIS — Teste Hold-out")
print("="*70)
print(df_teste.to_string(index=False))

# ── Melhor modelo por target ──────────────────────────────────────────────
print("\n=== Melhor modelo por target (R² no teste) ===")
melhores = {}
for target in TARGETS:
    melhor = df_teste[df_teste['Target']==target].sort_values('R2_teste', ascending=False).iloc[0]
    melhores[target] = melhor['Algoritmo']
    bateu = "✅" if melhor['Bateu_baseline'] else "❌ NÃO"
    print(f"  {target}: {melhor['Algoritmo']} | R²={melhor['R2_teste']:.3f} "
          f"| SMAPE={melhor['SMAPE_teste']:.1%} | Bateu baseline: {bateu}")

# ── Salvar tudo ───────────────────────────────────────────────────────────
df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv', index=False)
df_teste.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl', 'wb') as f:   pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl', 'wb') as f:  pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl', 'wb') as f:       pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)

relatorio = {
    'versao'           : 'V1_nested_groupkfold',
    'n_splits_ext'     : N_SPLITS_EXT,
    'n_splits_int'     : N_SPLITS_INT,
    'algoritmos'       : list(ALGORITMOS.keys()),
    'targets'          : TARGETS,
    'features'         : FEATURES,
    'log_targets'      : list(LOG_TARGETS),
    'baselines'        : {t: {k: float(v) for k, v in b.items() if isinstance(v, (int, float, np.floating))}
                           for t, b in baselines.items()},
    'melhores'         : melhores,
    'resultados_cv'    : [{k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                            for k, v in r.items()} for r in rows_cv],
    'resultados_teste' : [{k: (float(v) if isinstance(v, (int, float, np.floating)) else
                               bool(v) if isinstance(v, (bool, np.bool_)) else v)
                            for k, v in r.items()} for r in rows_teste],
    'feature_importances': {t: {'algoritmo': v['algoritmo'],
                                'top10': dict(list(v['importancias'].items())[:10])}
                             for t, v in feature_importances.items()},
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

logger.info("Script 3 concluído | modelos salvos: %d", len(TARGETS) * len(ALGORITMOS))

print("\n" + "═"*70)
print("  RESUMO FINAL — Script 3 (Modelagem)")
print("═"*70)
print(f"  Modelos treinados : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  CV estratégia     : GroupKFold(n={N_SPLITS_EXT}) por empresa — sem leakage temporal")
print(f"  Log-transform     : {list(LOG_TARGETS)}")
print(f"  Artefatos salvos  : resultados_cv.csv, resultados_teste.csv,")
print(f"                      relatorio_modelagem.json, feature_importance.png,")
print(f"                      curvas_aprendizado.png, analise_residuos.png")
print("═"*70)
print("  ✅ Pronto para o Script 4 (interpretabilidade / SHAP)")
print("═"*70)